# AeroSynth-Eval development batch on Kaggle
This produces real development-set model outputs, not human-alignment or protected-test evidence.

In [ ]:
import subprocess

!nvidia-smi
gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True,
).strip()
print("GPU:", gpu)
assert "P100" in gpu, "Choose GPU P100 in Kaggle settings before continuing."

In [ ]:
import os
from pathlib import Path

ROOT = Path("/kaggle/working/AeroSynth-Eval")
if not ROOT.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/triasha72/AeroSynth-Eval.git", str(ROOT)],
        check=True,
    )
os.chdir(ROOT)
# Kaggle's newest PyTorch wheels omit Pascal/sm_60 kernels required by P100.
subprocess.run(
    [
        "python",
        "-m",
        "pip",
        "install",
        "-q",
        "--force-reinstall",
        "--index-url",
        "https://download.pytorch.org/whl/cu126",
        "torch==2.7.1",
        "torchvision==0.22.1",
        "torchaudio==2.7.1",
    ],
    check=True,
)
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", ".[kaggle]"], check=True)
subprocess.run(
    [
        "python",
        "-c",
        "import torch; print(torch.__version__, torch.version.cuda); "
        "assert torch.cuda.get_device_capability(0) == (6, 0)",
    ],
    check=True,
)

In [ ]:
OUTPUT = Path("/kaggle/working/aerosynth-vlm-batch")
base_command = [
    "aerosynth-eval",
    "run-vlm-batch",
    "--backend",
    "transformers",
    "--model",
    "Qwen/Qwen2-VL-2B-Instruct",
    "--max-tokens",
    "800",
    "--temperature",
    "0",
    "--max-retries",
    "1",
    "--output-directory",
    str(OUTPUT),
]
subprocess.run([*base_command, "--dry-run"], check=True)

In [ ]:
subprocess.run(base_command, check=True)

In [ ]:
import hashlib
import tarfile

records = sorted(OUTPUT.glob("vlm-batch-*.json"))
assert records, "The real batch did not produce a provenance record."
archive = Path("/kaggle/working/aerosynth-vlm-batch.tar.gz")
with tarfile.open(archive, "w:gz") as tar:
    tar.add(OUTPUT, arcname=OUTPUT.name)
print("record:", records[-1])
print("archive_sha256:", hashlib.sha256(archive.read_bytes()).hexdigest())